In [15]:
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [ ]:
SYNTHETIC_SAMPLES = 10000
METADATA_FILE = "spoofing_metadata.json"


In [ ]:
from gps_ids import schema

_schema = schema.get("uav")
X, y = schema.load_real(_schema)
df = X.copy()
df[_schema.label_column] = y


In [19]:
# Load or create metadata
def _columns_of(meta):
    d = meta.to_dict()
    table = list(d['tables'].values())[0] if 'tables' in d else d
    return set(table['columns'])


def _detect(frame):
    meta = Metadata()
    meta.detect_table_from_dataframe(data=frame, table_name='spoofing_data')
    return meta


metadata = None
if os.path.exists(METADATA_FILE):
    cached = Metadata.load_from_json(METADATA_FILE)
    if _columns_of(cached) == set(df.columns):
        metadata = cached
        print(f"Loaded metadata from {METADATA_FILE}")
    else:
        print(f"{METADATA_FILE} does not describe this dataframe; re-detecting.")

if metadata is None:
    metadata = _detect(df)
    metadata.save_to_json(METADATA_FILE, mode='overwrite')
    print(f"Created and saved new metadata to {METADATA_FILE}")

Created and saved new metadata to spoofing_metadata.json


In [20]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        # Uses GAN-based, deep learning methods to train a model and generate synthetic data.
        'name': 'CTGAN',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 4000,
            'verbose': True,
            'cuda': True
        },
        'output': str(schema.synthetic_path(_schema, "spoofing-CTGAN-synthetic.csv")),
        'pickle': "spoofing-CTGAN-synthesizer.pkl"
    },
    {
        # Uses classic, statistical methods to train a model and generate synthetic data
        'name': 'GaussianCopula',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'output': str(schema.synthetic_path(_schema, "spoofing-GaussianCopula-synthetic.csv")),
        'pickle': "spoofing-GaussianCopula-synthesizer.pkl"
    },
    {
        # Uses a variational autoencoder (VAE)-based, neural network techniques to train a model and generate synthetic data.
        'name': 'TVAE',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'output': str(schema.synthetic_path(_schema, "spoofing-TVAE-synthetic.csv")),
        'pickle': "spoofing-TVAE-synthesizer.pkl"
    }
]

In [21]:
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthesizer.reset_sampling()
    synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
    print(f"Synthetic data preview for {synth['name']}:")
    print(synthetic_data.head())
    synthetic_data.to_csv(synth['output'], index=False)
    print(f"Synthetic dataset saved to {synth['output']}")
    synthesizer.save(filepath=synth['pickle'])
    print(f"Trained synthesizer saved to {synth['pickle']}")


Processing CTGAN...


Gen. (0.87) | Discrim. (-0.14): 100%|██████████| 4000/4000 [4:03:18<00:00,  3.65s/it]   


Synthetic data preview for CTGAN:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.386859  0.014702  0.008626  0.934173          0.807294          0.118945   
1 -1.060172  0.015740 -0.206287  0.192483          0.997379          0.070111   
2 -0.759315 -0.030135 -0.070877  0.870946          1.388260          0.104090   
3 -0.377439  0.092694  0.128247  0.845542          0.926242          0.004465   
4 -0.375547  0.190939 -0.040513  0.914830          0.777948         -0.016679   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0          0.070934          0.073471            3.146717  36.205010  ...   
1          0.116384         -0.308859            3.290596  36.204807  ...   
2         -0.222300          0.224828            3.388882  36.204807  ...   
3         -0.067507         -0.228298            3.607470  36.204807  ...   
4          0.020419         -0.061021            3.022755  36.204807  ...   

   v_z_valid  xy

Loss: 3.678: 100%|██████████| 2000/2000 [48:43<00:00,  1.46s/it] 


Synthetic data preview for TVAE:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.393468  0.018125 -0.035691  0.924019          1.057982         -0.063401   
1 -0.371375  0.005782 -0.065307  0.929230          0.706263          0.025533   
2 -0.377207 -0.007014 -0.039199  0.844494          1.051712         -0.117694   
3 -0.366291 -0.017526 -0.092557  0.909152          1.087979         -0.001071   
4 -0.401563  0.017426  0.071095  0.938339          1.172002         -0.028700   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0          0.083753          0.047320            3.131089  36.204814  ...   
1          0.307709          0.027385            2.991147  36.204835  ...   
2         -0.059093         -0.128909            3.047150  36.204847  ...   
3          0.041447          0.030771            2.960551  36.204837  ...   
4          0.300199         -0.177873            2.984924  36.204820  ...   

   v_z_valid  xy_